# FUT.AI — Fase 5 · Control de calidad de métricas (MVP local)

**Propósito:** auditar las salidas de la Fase 4 antes de conectar los datos a la API.

- No requiere GPU, video ni ejecutar YOLO/ByteTrack de nuevo.
- No modifica las métricas originales: genera reportes diagnósticos y un JSON de intercambio con la web.
- **Importante:** ningún indicador estadístico demuestra por sí mismo precisión en metros ni identifica futbolistas reales. Todas las métricas se mantienen como **provisionales**.

**Si sigues en la misma sesión de Colab, el notebook buscará los archivos en la carpeta anterior. Si reiniciaste Colab, sube estos cuatro archivos:** `posiciones_dinamicas.csv`, `metricas_por_track_provisionales.csv`, `metricas_tramos_provisionales.csv`, `reporte_calidad.json`. Después ejecuta las celdas en orden.

## Paso 1 — Cargar resultados de la fase anterior

In [ ]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path(os.getenv('FUTAI_INPUT_DIR', '/content'))
archivos_requeridos = ['posiciones_dinamicas.csv','metricas_por_track_provisionales.csv',
                       'metricas_tramos_provisionales.csv','reporte_calidad.json']
if not all((BASE / nombre).exists() for nombre in archivos_requeridos):
    alternativa = BASE / 'futai' / 'fase4_dinamica'
    if all((alternativa / nombre).exists() for nombre in archivos_requeridos):
        BASE = alternativa
SALIDA = BASE / 'fase5_calidad'
SALIDA.mkdir(parents=True, exist_ok=True)

archivos = {
    'pos': 'posiciones_dinamicas.csv',
    'resumen': 'metricas_por_track_provisionales.csv',
    'tramos': 'metricas_tramos_provisionales.csv',
    'calidad': 'reporte_calidad.json',
}
for nombre, archivo in archivos.items():
    if not (BASE / archivo).exists():
        raise FileNotFoundError(f'Falta {archivo} en {BASE}. Súbelo desde el panel Archivos de Colab.')

pos = pd.read_csv(BASE / archivos['pos'])
resumen = pd.read_csv(BASE / archivos['resumen'])
tramos = pd.read_csv(BASE / archivos['tramos'])
reporte = json.loads((BASE / archivos['calidad']).read_text(encoding='utf-8'))
fps = float(reporte['fps'])
nframes = int(reporte['frames'])
cols = {'frame','tiempo_s','track_id','x_m','y_m','calibracion_valida','en_cancha','ancla'}
if not cols.issubset(pos.columns):
    raise ValueError(f'Faltan columnas de posiciones: {cols-set(pos.columns)}')
print('Video:', reporte['video'])
print('Frames:', nframes, '| FPS:', fps)
print('Cobertura de homografía:', reporte['porcentaje_frames_con_homografia'], '%')
print('Observaciones:', len(pos), '| En cancha/calibradas:', int(pos.en_cancha.sum()))
print('Tracks con métricas provisionales:', len(resumen), '| Tramos:', len(tramos))

## Paso 2 — Identificar huecos y saltos sospechosos

El cálculo usa **diferencias entre fotogramas consecutivos** antes del suavizado. Sirve para detectar inestabilidad en las coordenadas, no para estimar la velocidad final de los jugadores.


In [ ]:
# Fotogramas sin una homografía válida
frames_validos = set(pos.loc[pos.calibracion_valida.astype(bool), 'frame'].astype(int).unique())
frames_faltantes = [f for f in range(nframes) if f not in frames_validos]

huecos = []
for f in frames_faltantes:
    if not huecos or f > huecos[-1][-1]+1:
        huecos.append([f])
    else:
        huecos[-1].append(f)
tabla_huecos = pd.DataFrame([
    {'frame_inicio':g[0], 'frame_fin':g[-1], 'segundo_inicio':round(g[0]/fps,2),
     'segundo_fin':round(g[-1]/fps,2), 'frames':len(g)}
    for g in huecos
])
print('INTERVALOS SIN HOMOGRAFÍA')
print(tabla_huecos.to_string(index=False) if not tabla_huecos.empty else 'Ninguno')

# No conectar gaps de detección ni cambios de ancla (calibración de referencia)
validas = (pos.loc[pos.calibracion_valida.astype(bool) & pos.en_cancha.astype(bool)]
             .sort_values(['track_id','frame']).copy())
validas['dt_frames'] = validas.groupby('track_id').frame.diff()
validas['dx'] = validas.groupby('track_id').x_m.diff()
validas['dy'] = validas.groupby('track_id').y_m.diff()
validas['ancla_previa'] = validas.groupby('track_id').ancla.shift(1)
validas['mismo_ancla'] = validas.ancla.eq(validas.ancla_previa)
secuencia = validas[validas.dt_frames.eq(1) & validas.mismo_ancla].copy()
secuencia['velocidad_bruta_kmh'] = (
    3.6 * fps * np.hypot(secuencia.dx, secuencia.dy)
)
secuencia['sospechoso_mayor_40'] = secuencia.velocidad_bruta_kmh > 40
secuencia['sospechoso_mayor_60'] = secuencia.velocidad_bruta_kmh > 60
print('\nIntervalos consecutivos revisados:',len(secuencia))
print('Saltos >40 km/h:',int(secuencia.sospechoso_mayor_40.sum()),
      f'({100*secuencia.sospechoso_mayor_40.mean():.1f} %)')
print('Saltos >60 km/h:',int(secuencia.sospechoso_mayor_60.sum()))
print('Mediana velocidad bruta (km/h):',round(secuencia.velocidad_bruta_kmh.median(),2))

segundo = np.floor(np.arange(nframes)/fps).astype(int)
calidad_segundos = pd.DataFrame({'frame':np.arange(nframes), 'segundo':segundo})
calidad_segundos['con_homografia'] = calidad_segundos.frame.isin(frames_validos)
cobertura_seg = calidad_segundos.groupby('segundo').con_homografia.agg(['sum','count']).reset_index()
cobertura_seg['cobertura_pct'] = 100*cobertura_seg['sum']/cobertura_seg['count']
secuencia['segundo'] = np.floor(secuencia.tiempo_s).astype(int)
flujos = secuencia.groupby('segundo').agg(intervalos=('track_id','size'),
                        saltos_40=('sospechoso_mayor_40','sum'),
                        saltos_60=('sospechoso_mayor_60','sum')).reset_index()
calidad_segundos = cobertura_seg.merge(flujos, on='segundo', how='left').fillna(0)
calidad_segundos['porcentaje_saltos_40'] = np.where(
    calidad_segundos.intervalos > 0,
    100*calidad_segundos.saltos_40/calidad_segundos.intervalos, np.nan
)
calidad_segundos['bandera'] = np.select([
    calidad_segundos.cobertura_pct<90,
    calidad_segundos.porcentaje_saltos_40>10
], ['REVISAR_COBERTURA','REVISAR_SALTOS'], default='INSPECCION_VISUAL')
calidad_segundos.to_csv(SALIDA/'calidad_por_segundo.csv',index=False)
tabla_huecos.to_csv(SALIDA/'huecos_homografia.csv',index=False)
print('\nCalidad por segundo:')
print(calidad_segundos[['segundo','cobertura_pct','intervalos','porcentaje_saltos_40','bandera']].to_string(index=False))

## Paso 3 — Ver gráficos de calidad

In [ ]:
fig, ax = plt.subplots(figsize=(11,3.5))
ax.plot(calidad_segundos.segundo, calidad_segundos.cobertura_pct, marker='o')
ax.set(ylim=(-5,105), xlabel='Segundo de video', ylabel='Frames con homografía (%)',
       title='Cobertura de calibración a lo largo del video')
ax.set_xticks(calidad_segundos.segundo)
ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(SALIDA/'cobertura_temporal.png',dpi=150)
plt.show()

fig, ax=plt.subplots(figsize=(11,3.5))
ax.plot(calidad_segundos.segundo, calidad_segundos.porcentaje_saltos_40, marker='o')
ax.axhline(10,ls='--',label='Umbral de inspección (heurístico)')
ax.set(xlabel='Segundo de video',ylabel='Intervalos anómalos (%)',
       title='Saltos de posición equivalentes a más de 40 km/h (sin suavizado)')
ax.set_xticks(calidad_segundos.segundo)
ax.grid(True,alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(SALIDA/'saltos_por_segundo.png',dpi=150)
plt.show()

## Paso 4 — Priorizar tracks para inspección humana

Estas banderas **no validan** la precisión deportiva. Identifican trayectorias que conviene observar antes de presentar una velocidad o distancia.


In [ ]:
revision = resumen.copy()
if not revision.empty:
    revision['intervalos_descartados'] = revision.track_id.map(
        tramos.groupby('track_id').intervalos_descartados.sum()
    ).fillna(0).astype(int)
    revision['observaciones'] = revision.track_id.map(
        validas.groupby('track_id').size()
    ).fillna(0).astype(int)
    revision['revision_recomendada'] = np.where(
        (revision.duracion_observada_s < 5) |
        (revision.numero_tramos > 2) |
        (revision.intervalos_descartados >= 2) |
        (revision.velocidad_maxima_estimada_kmh >= 35),
        'PRIORITARIA', 'NORMAL'
    )
    revision['estado'] = 'PROVISIONAL_NO_VALIDADO'
    revision = revision.sort_values(['revision_recomendada','distancia_observada_m'],ascending=[False,False])
    print(revision[['track_id','distancia_observada_m','velocidad_maxima_estimada_kmh',
          'numero_tramos','intervalos_descartados','revision_recomendada']].to_string(index=False))
    print('\nTracks para revisión prioritaria:',int((revision.revision_recomendada=='PRIORITARIA').sum()))
    revision.to_csv(SALIDA/'metricas_con_banderas.csv',index=False)
else:
    print('No hay tracks con métricas para analizar.')

## Paso 5 — Exportar JSON para el backend de FUT.AI

Se entrega **un contrato de intercambio provisional**, no resultados certificados. El backend podrá guardar y mostrar estos campos con la etiqueta `PROVISIONAL_NO_VALIDADO` mientras mejora la homografía.


In [ ]:
def numero(v):
    return None if pd.isna(v) else float(v)

salida_api = {
    'video': reporte['video'],
    'procesamiento': {
        'tipo': 'analisis_video_exploratorio',
        'frames': nframes,
        'fps': fps,
        'cobertura_homografia_pct': numero(reporte['porcentaje_frames_con_homografia']),
        'observaciones_tracking': int(reporte['observaciones_tracking']),
        'observaciones_en_cancha_y_calibradas': int(reporte['observaciones_en_cancha_y_calibradas']),
        'estado': 'PROVISIONAL_NO_VALIDADO'
    },
    'metricas_por_track': [
        {'track_id':int(r.track_id),
         'distancia_observada_m':numero(r.distancia_observada_m),
         'velocidad_maxima_estimada_kmh':numero(r.velocidad_maxima_estimada_kmh),
         'duracion_observada_s':numero(r.duracion_observada_s),
         'numero_tramos':int(r.numero_tramos),
         'revision_recomendada':r.revision_recomendada,
         'estado':'PROVISIONAL_NO_VALIDADO'}
        for r in revision.itertuples(index=False)
    ],
    'advertencias': reporte.get('advertencias',[])+[
        'El cálculo no permite afirmar distancia total o velocidad máxima real de un futbolista.',
        'Un track_id representa una trayectoria temporal, no necesariamente una identidad permanente.',
    ]
}
json_path = SALIDA/'resultados_para_backend_PROVISIONALES.json'
json_path.write_text(json.dumps(salida_api,ensure_ascii=False,indent=2),encoding='utf-8')
print('Archivos generados en:',SALIDA)
for f in sorted(SALIDA.iterdir()):
    print('  -',f.name)
print('\nSIGUIENTE ACCIÓN: revisar los segundos marcados REVISAR_COBERTURA/REVISAR_SALTOS',
      'y corroborar trayectorias contra el video original antes de llamar reales a las métricas.')